0️⃣  **Check individual samples**
1️⃣  Compare replicates
2️⃣  Compare materials

# 0️⃣  Check individual samples - OER Analysis (HZB CE-NESD group)

The calculations and desired plots are based on information provided by Bastian Schmiedecke.

In [1]:
%%capture
%matplotlib widget
#!pip install requests_cache

import matplotlib.pyplot as plt
import time
import requests
import pandas as pd
import numpy as np
from datetime import datetime

import plotly.graph_objects as go
import plotly.io as pio
import json

from scipy.signal import find_peaks

import ipywidgets as widgets
from IPython.display import display

import sys
sys.path.insert(1, '../python-scripts-c6fxKDJrSsWp1xCxON1Y7g')
sys.path.insert(1, '../../python-scripts-c6fxKDJrSsWp1xCxON1Y7g')
from api_calls import *
from plotly_helpers import *

url = "https://nomad-hzb-ce.de/nomad-oasis/api/v1"

import os
token = os.environ['NOMAD_CLIENT_ACCESS_TOKEN']

In [2]:
def get_ids_of_all_samples(url, token, author_name, sample_type='CE_NOME_Sample'):
    row = {"entry_type":sample_type,
           #"authors.name": "Abdelrahman Ishmael"
           #"authors.name": "Sofia Muñoz"
           #"authors.name": "Felipe Mata mata"
           "authors.name": author_name
        }
    query = {
        'required': {
            'metadata': '*',
            'data': '*',
        },
        'owner': 'visible',
        'query': row,
        'pagination': {
            'page_size': 100
        }
    }
    response = requests.post(f'{url}/entries/archive/query',
                             headers={'Authorization': f'Bearer {token}'}, json=query)
    return [s["archive"]["data"].get("lab_id") for s in response.json()["data"] if s["archive"]["data"].get("lab_id")]

# TODO get all nesd grouped by folder or sampe or upload

In [3]:
# all ipywidgets
uploads = get_all_uploads(url, token, number_of_uploads=200)
upload_names = [u.get("upload_name","--no-name--") for u in uploads]

upload_selector = widgets.Dropdown(
    options=['-- please select an upload --'] + upload_names,
    value='-- please select an upload --',
    description='NOMAD upload:',
    style={'description_width': 'initial'}
)

folder_selector = widgets.Dropdown(
    description="Select Folder:",
    style={'description_width': 'initial'}
)

show_sample_details_button = widgets.Button(
    description="Show plots for selected sample ID",
    button_style='info',
    layout=widgets.Layout(width='auto')
)

overview_output = widgets.Output()
folder_select_output = widgets.Output()
analysis_output = widgets.Output()

out1 = widgets.Output()
out2 = widgets.Output()
out3 = widgets.Output()
out4 = widgets.Output()

tab = widgets.Tab(children=[out1, out2, out3, out4])
tab.set_title(0, 'CV')
tab.set_title(1, 'LSV')
tab.set_title(2, 'EIS')
tab.set_title(3, 'CA')

## Choose the NOMAD upload you want to analyse

In [4]:
def on_upload_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        overview_output.clear_output()
        folder_select_output.clear_output()
        analysis_output.clear_output()
        with overview_output:
            upload_name = change['new']
            cv_entries = get_specific_entrytype_of_uploads(url, token, [upload_name], 'CE_NESD_CyclicVoltammetry', with_meta=False)
            ca_entries = get_specific_entrytype_of_uploads(url, token, [upload_name], 'CE_NESD_Chronoamperometry', with_meta=False)
            eis_entries = get_specific_entrytype_of_uploads(url, token, [upload_name], 'CE_NESD_PEIS', with_meta=False)
            lsv_entries = get_specific_entrytype_of_uploads(url, token, [upload_name], 'CE_NESD_LinearSweepVoltammetry', with_meta=False)
            entry_names = [entry.get('name') for entry in cv_entries]
            folder_names = sorted({s.rsplit("/", 1)[0] for s in entry_names})
            print(f'Number of CV entries: {len(cv_entries)}')
            print(f'Number of CA entries: {len(ca_entries)}')
            print(f'Number of EIS entries: {len(eis_entries)}')
            print(f'Number of LSV entries: {len(lsv_entries)}')

        if len(folder_names) != 0:
            folder_selector.options = folder_names
            with folder_select_output:
                folder_select_output.clear_output()
                display(folder_selector)

        if upload_selector.value == '-- please select an upload --':
            with folder_select_output:
                print('Please select an upload before inspecting individual runs.')

upload_selector.observe(on_upload_change)

display(upload_selector, overview_output)

Dropdown(description='NOMAD upload:', options=('-- please select an upload --', 'NESD Showcase OER Analysis - …

Output()

## Inspect the data

In [5]:
def get_cv_plot(cv_entry):
    name = cv_entry.get('name')
    cycles = cv_entry.get('cycles')
    current_list = [cycle.get('current') for cycle in cycles]
    voltage_list = [cycle.get('voltage') for cycle in cycles]
    label_list = [f'cycle {idx + 1}' for idx in range(len(cycles))]
    fig = get_line_plot(f'Current over Voltage {name}',  voltage_list, 'Voltage [V]', current_list, 'Current [mA]', label_list)
    return fig

def get_lsv_plot(lsv_entry):
    name = lsv_entry.get('name')
    current_list = [lsv_entry.get('current')]
    voltage_list = [lsv_entry.get('voltage')]
    label_list = ['']
    fig = get_line_plot(f'Current over Voltage {name}',  voltage_list, 'Voltage [V]', current_list, 'Current [mA]', label_list)
    return fig

def get_first_plot_from_entry(entry):
    plot_dict = entry.get('figures')[0].get('figure')
    fig = pio.from_json(pio.to_json(plot_dict))
    return fig

def get_second_plot_from_entry(entry):
    plot_dict = entry.get('figures')[1].get('figure')
    fig = pio.from_json(pio.to_json(plot_dict))
    return fig
    
def get_lsv_plot_from_entry(entry):
    return get_second_plot_from_entry(entry)

def get_nyquist_plot_from_entry(entry):
    return get_first_plot_from_entry(entry)

def get_bode_plot_from_entry(entry):
    return get_second_plot_from_entry(entry)

In [6]:
def get_filtered_entries(upload_name, entry_type, folder_path):
    entries = get_specific_entrytype_of_uploads(url, token, [upload_name], entry_type, with_meta=False)
    entries = [entry for entry in entries if entry.get('name').startswith(folder_path)]
    entries = sorted(
        entries,
        key=lambda e: datetime.fromisoformat(e.get('datetime'))
    )
    return entries

def on_folder_idx_change(change):
    if change['name'] == 'value' and change['new'] is not None:
        folder_idx = change['new']
        with folder_select_output:
            folder_select_output.clear_output()
            analysis_output.clear_output()
            display(folder_selector)
        with out1:
            out1.clear_output()
            cv_entries = get_filtered_entries(upload_selector.value, 'CE_NESD_CyclicVoltammetry', folder_idx)
            for cv_entry in cv_entries:
                fig = get_cv_plot(cv_entry)
                fig.show()
        with out2:
            out2.clear_output()
            lsv_entries = get_filtered_entries(upload_selector.value, 'CE_NESD_LinearSweepVoltammetry', folder_idx)
            for lsv_entry in lsv_entries:
                print(lsv_entry.get('name'))
                fig2 = get_lsv_plot_from_entry(lsv_entry)
                fig2.show()
        with out3:
            out3.clear_output()
            eis_entries = get_filtered_entries(upload_selector.value, 'CE_NESD_PEIS', folder_idx)
            for eis_entry in eis_entries:
                print(eis_entry.get('name'))
                fig1 = get_nyquist_plot_from_entry(eis_entries[0])
                fig2 = get_bode_plot_from_entry(eis_entries[0])
                fig1.show()
                fig2.show()
        with out4:
            out4.clear_output()
            ca_entries = get_filtered_entries(upload_selector.value, 'CE_NESD_Chronoamperometry', folder_idx)
            for ca_entry in ca_entries:
                print(ca_entry.get('name'))
                fig = get_first_plot_from_entry(ca_entry)
                fig.show()
        with analysis_output:
            analysis_output.clear_output()
            display(tab)

folder_selector.observe(on_folder_idx_change, names='value')

if upload_selector.value == '-- please select an upload --':
    with folder_select_output:
        print('Please select an upload before inspecting individual runs.')

display(folder_select_output, analysis_output)


Output()

Output()